# KV Cache 量化 —— 为什么 W4 不一定是推理的最大瓶颈

对应文章：《大模型量化算法（24）：KV Cache 量化》
https://lrypcy.github.io/2026/09/19/llm-quant-24-kv-cache/

| 实验 | 问题 |
|---|---|
| A | K/V 量化误差如何放大成注意力输出误差？K 和 V 谁更敏感？粒度选哪个？ |
| B | 误差随序列长度如何累积（KV 会被反复读取）？ |
| C | 显存/带宽账本：KV cache 字节 vs 权重字节，随 batch/seq 谁先成为瓶颈？ |

纯 numpy 合成注意力，SEED=0。

In [1]:
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 0
MODE = "smoke"
CFG = {
    "smoke": dict(seqs=(128, 512), bits=(2, 4, 8), sweep=40),
    "full":  dict(seqs=(128, 512, 2048, 8192), bits=(2, 3, 4, 6, 8), sweep=120),
}[MODE]
HERE = os.getcwd(); RES = os.path.join(HERE, "results"); os.makedirs(RES, exist_ok=True)
_LINES = []
def log(m=""):
    print(m); _LINES.append(str(m))
def savefig(fig, name):
    p = os.path.join(RES, name); fig.savefig(p, dpi=130, bbox_inches="tight"); plt.close(fig)
    log(f"[save] {p}"); return p
def rel_mse(a, b):
    return float(np.sum((a - b) ** 2) / np.sum(a ** 2))
log(f"MODE={MODE} CFG={CFG}")

MODE=smoke CFG={'seqs': (128, 512), 'bits': (2, 4, 8), 'sweep': 40}


In [2]:
def make_attention(n_head=4, d_head=32, seq=512, seed=SEED, heavy_tail=True):
    r = np.random.default_rng(seed)
    Q = r.normal(0, 1, (seq, n_head * d_head))
    K = r.normal(0, 1, (seq, n_head * d_head))
    V = r.normal(0, 1, (seq, n_head * d_head))
    if heavy_tail:                                    # 制造 K/V 的离群通道（LLM 的真实形态）
        idx = r.choice(n_head * d_head, 8, replace=False)
        K[:, idx] *= 8.0
        V[:, idx] *= 4.0
    return Q, K, V


def attention(Q, K, V, n_head=4):
    seq, d = Q.shape
    dh = d // n_head
    Qh = Q.reshape(seq, n_head, dh).transpose(1, 0, 2)
    Kh = K.reshape(seq, n_head, dh).transpose(1, 0, 2)
    Vh = V.reshape(seq, n_head, dh).transpose(1, 0, 2)
    scale = 1.0 / np.sqrt(dh)
    S = Qh @ Kh.transpose(0, 2, 1) * scale
    S = S - S.max(axis=-1, keepdims=True)
    P = np.exp(S); P = P / P.sum(axis=-1, keepdims=True)
    O = P @ Vh
    return O.transpose(1, 0, 2).reshape(seq, d)


def quantize(X, b, granularity="per-tensor"):
    qmax = 2 ** (b - 1) - 1
    if granularity == "per-tensor":
        s = max(np.max(np.abs(X)) / qmax, 1e-12)
        return s * np.clip(np.round(X / s), -qmax - 1, qmax)
    if granularity == "per-channel":            # 按隐维（列）
        s = np.maximum(np.max(np.abs(X), axis=0, keepdims=True) / qmax, 1e-12)
        return s * np.clip(np.round(X / s), -qmax - 1, qmax)
    if granularity == "per-token":              # 按 token（行）
        s = np.maximum(np.max(np.abs(X), axis=1, keepdims=True) / qmax, 1e-12)
        return s * np.clip(np.round(X / s), -qmax - 1, qmax)
    raise ValueError(granularity)


SEQ = CFG["seqs"][-1]
Q, K, V = make_attention(seq=SEQ)
O = attention(Q, K, V)
log(f"attention: seq={SEQ}, d={Q.shape[1]}, heads=4；K/V 有 8 个离群通道")

attention: seq=512, d=128, heads=4；K/V 有 8 个离群通道


In [3]:
rows_A = []
for b in CFG["bits"]:
    for gran in ("per-tensor", "per-channel", "per-token"):
        for target in ("K", "V", "KV"):
            Kq = quantize(K, b, gran) if target in ("K", "KV") else K
            Vq = quantize(V, b, gran) if target in ("V", "KV") else V
            Oq = attention(Q, Kq, Vq)
            rows_A.append(dict(bits=b, gran=gran, target=target, out=rel_mse(O, Oq)))

log("=" * 84)
log(f"[A] KV 量化 -> 注意力输出误差（seq={SEQ}）")
log(f"{'bits':>5} {'granularity':>13} {'只量化K':>12} {'只量化V':>12} {'K+V':>12}")
for b in CFG["bits"]:
    for gran in ("per-tensor", "per-channel", "per-token"):
        g = lambda t: [r["out"] for r in rows_A if r["bits"] == b and r["gran"] == gran and r["target"] == t][0]
        log(f"{b:>5} {gran:>13} {g('K'):>12.3e} {g('V'):>12.3e} {g('KV'):>12.3e}")
b4 = CFG["bits"][-2] if 4 in CFG["bits"] else CFG["bits"][-1]
for b in (b4,):
    for gran in ("per-tensor", "per-channel", "per-token"):
        g = lambda t: [r["out"] for r in rows_A if r["bits"] == b and r["gran"] == gran and r["target"] == t][0]
        log(f"  {b}-bit/{gran}：V 的误差是 K 的 {g('V')/g('K'):.2f} 倍（V 直接参与加权求和，通常更敏感）")
pt = [r for r in rows_A if r["bits"] == b4 and r["gran"] == "per-tensor" and r["target"] == "KV"][0]
pc = [r for r in rows_A if r["bits"] == b4 and r["gran"] == "per-channel" and r["target"] == "KV"][0]
log(f"  {b}-bit 粒度收益（KV 同时量化）：per-tensor {pt['out']:.3e} -> per-channel {pc['out']:.3e}，"
    f"{10*np.log10(pt['out']/pc['out']):+.2f} dB")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for gran, c in (("per-tensor", "#C44E52"), ("per-token", "#DD8452"), ("per-channel", "#4C72B0")):
    sub = [r for r in rows_A if r["gran"] == gran and r["target"] == "KV"]
    ax[0].plot([r["bits"] for r in sub], [r["out"] for r in sub], "o-", lw=2, color=c, label=gran)
ax[0].set_yscale("log"); ax[0].invert_xaxis()
ax[0].set_xlabel("KV bits"); ax[0].set_ylabel("attention output rel. MSE")
ax[0].set_title("[A] KV cache quantization: granularity matters"); ax[0].legend(fontsize=9)

sub = [r for r in rows_A if r["bits"] == b4]
labels = [f"{r['target']}\n{r['gran']}" for r in sub]
ax[1].bar(range(len(sub)), [r["out"] for r in sub], color="#4C72B0")
ax[1].set_yscale("log"); ax[1].set_xticks(range(len(sub)))
ax[1].set_xticklabels(labels, fontsize=6.5, rotation=45, ha="right")
ax[1].set_ylabel("attention output rel. MSE")
ax[1].set_title(f"[A] {b4}-bit: K vs V vs KV")
savefig(fig, "kv_cache_sensitivity.png")

[A] KV 量化 -> 注意力输出误差（seq=512）
 bits   granularity         只量化K         只量化V          K+V
    2    per-tensor    1.216e+00    8.992e-01    1.116e+00
    2   per-channel    1.571e+00    6.599e-01    2.442e+00
    2     per-token    8.333e-01    6.127e-01    1.182e+00
    4    per-tensor    3.893e-01    1.799e-01    5.817e-01
    4   per-channel    4.673e-02    1.697e-02    6.344e-02
    4     per-token    2.650e-01    4.914e-02    3.164e-01
    8    per-tensor    2.136e-03    5.908e-04    2.716e-03
    8   per-channel    2.185e-04    5.358e-05    2.775e-04
    8     per-token    1.068e-03    1.500e-04    1.222e-03
  4-bit/per-tensor：V 的误差是 K 的 0.46 倍（V 直接参与加权求和，通常更敏感）
  4-bit/per-channel：V 的误差是 K 的 0.36 倍（V 直接参与加权求和，通常更敏感）
  4-bit/per-token：V 的误差是 K 的 0.19 倍（V 直接参与加权求和，通常更敏感）
  4-bit 粒度收益（KV 同时量化）：per-tensor 5.817e-01 -> per-channel 6.344e-02，+9.62 dB


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/kv_cache_quant/results/kv_cache_sensitivity.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/kv_cache_quant/results/kv_cache_sensitivity.png'

In [4]:
rows_B = []
for seq in CFG["seqs"]:
    Qs, Ks, Vs = make_attention(seq=seq)
    Os = attention(Qs, Ks, Vs)
    for b in CFG["bits"]:
        Oq = attention(Qs, quantize(Ks, b, "per-channel"), quantize(Vs, b, "per-channel"))
        # 逐 token 的误差（后面的 token 会 attend 到更多已量化的 KV）
        err = np.sqrt(np.sum((Os - Oq) ** 2, axis=1)) / np.sqrt(np.sum(Os ** 2, axis=1))
        rows_B.append(dict(seq=seq, bits=b, out=rel_mse(Os, Oq),
                           first=float(err[:seq // 4].mean()), last=float(err[-seq // 4:].mean())))

log("=" * 84)
log("[B] 序列长度与误差累积（per-channel）")
log(f"{'seq':>7} {'bits':>5} {'output rel.MSE':>16} {'前25% token':>13} {'后25% token':>13}")
for r in rows_B:
    log(f"{r['seq']:>7} {r['bits']:>5} {r['out']:>16.3e} {r['first']:>13.3e} {r['last']:>13.3e}")
log("  读数：KV 量化误差不会随 token 位置单调放大（每个 token 的 attention 都是独立的加权平均），")
log("        但序列变长会让「被污染的 KV」总数增加 —— 长上下文才是 KV 量化的真正压力来源。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
for b in CFG["bits"]:
    sub = [r for r in rows_B if r["bits"] == b]
    ax[0].plot([r["seq"] for r in sub], [r["out"] for r in sub], "o-", lw=2, label=f"b={b}")
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("sequence length"); ax[0].set_ylabel("attention output rel. MSE")
ax[0].set_title("[B] Longer context = more quantized KV to read"); ax[0].legend(fontsize=9)

sub = [r for r in rows_B if r["bits"] == b4]
xs = np.arange(len(sub))
ax[1].bar(xs - 0.2, [r["first"] for r in sub], width=0.4, color="#4C72B0", label="first 25% tokens")
ax[1].bar(xs + 0.2, [r["last"] for r in sub], width=0.4, color="#C44E52", label="last 25% tokens")
ax[1].set_xticks(xs); ax[1].set_xticklabels([f"seq={r['seq']}" for r in sub], fontsize=8)
ax[1].set_ylabel("per-token rel. error"); ax[1].set_title(f"[B] {b4}-bit: error by token position")
ax[1].legend(fontsize=9)
savefig(fig, "kv_cache_sequence_scaling.png")

[B] 序列长度与误差累积（per-channel）
    seq  bits   output rel.MSE    前25% token    后25% token
    128     2        1.765e+00     1.347e+00     1.438e+00
    128     4        4.987e-02     2.562e-01     2.152e-01
    128     8        1.349e-04     1.460e-02     1.095e-02
    512     2        2.442e+00     1.767e+00     1.653e+00
    512     4        6.344e-02     2.597e-01     2.544e-01
    512     8        2.775e-04     1.635e-02     1.635e-02
  读数：KV 量化误差不会随 token 位置单调放大（每个 token 的 attention 都是独立的加权平均），
        但序列变长会让「被污染的 KV」总数增加 —— 长上下文才是 KV 量化的真正压力来源。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/kv_cache_quant/results/kv_cache_sequence_scaling.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/kv_cache_quant/results/kv_cache_sequence_scaling.png'

In [5]:
# ---------- C：显存/带宽账本（解析模型，不是实测） ----------
L, d_model, n_head, d_head = 32, 4096, 32, 128
params_per_layer = 4 * d_model ** 2          # qkv/o + mlp 的粗略量级
W_params = L * params_per_layer
kv_bytes_per_token = 2 * L * n_head * d_head  # K 和 V


def bytes_fmt(x):
    for u in ("B", "KB", "MB", "GB"):
        if x < 1024:
            return f"{x:.1f} {u}"
        x /= 1024
    return f"{x:.1f} TB"


rows_C = []
for batch in (1, 8, 32):
    for seq in (2048, 8192, 32768):
        kv = batch * seq * kv_bytes_per_token
        for wbits in (16, 4):
            wt = W_params * wbits / 8
            rows_C.append(dict(batch=batch, seq=seq, wbits=wbits, kv_bytes=kv, w_bytes=wt,
                               ratio=kv / wt))

log("=" * 84)
log(f"[C] 显存账本（模型：L={L}, d={d_model}, heads={n_head}；权重约 {W_params/1e9:.2f} B 参数）")
log(f"{'batch':>6} {'seq':>7} {'权重@4bit':>12} {'KV@fp16':>12} {'KV/权重':>9}")
for r in rows_C:
    if r["wbits"] == 4:
        log(f"{r['batch']:>6} {r['seq']:>7} {bytes_fmt(r['w_bytes']):>12} {bytes_fmt(r['kv_bytes']):>12} "
            f"{r['ratio']:>8.2f}x")
r1 = [r for r in rows_C if r["batch"] == 1 and r["seq"] == 2048 and r["wbits"] == 4][0]
r2 = [r for r in rows_C if r["batch"] == 32 and r["seq"] == 32768 and r["wbits"] == 4][0]
log(f"  batch=1/seq=2048：KV 是权重(4-bit)的 {r1['ratio']:.2f}x")
log(f"  batch=32/seq=32768：KV 是权重的 {r2['ratio']:.2f}x —— 这时权重位宽已经不是主要矛盾")
log("  读数：权重量化省的是「一次性」的显存与带宽，KV cache 随 batch×seq 线性增长；")
log("        长上下文 + 大 batch 下，KV 才是瓶颈 —— 这就是 24 篇的核心论点。")

fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))
sel = [r for r in rows_C if r["wbits"] == 4]
for batch in (1, 8, 32):
    sub = [r for r in sel if r["batch"] == batch]
    ax[0].plot([r["seq"] for r in sub], [r["ratio"] for r in sub], "o-", lw=2, label=f"batch={batch}")
ax[0].axhline(1.0, color="grey", ls="--", lw=1)
ax[0].set_xscale("log"); ax[0].set_yscale("log")
ax[0].set_xlabel("sequence length"); ax[0].set_ylabel("KV bytes / weight bytes (@W4)")
ax[0].set_title("[C] Where the memory actually goes"); ax[0].legend(fontsize=9)

sub = [r for r in rows_C if r["seq"] == 8192 and r["batch"] == 8]
ax[1].bar(["weights@16", "weights@4"], [sub[0]["w_bytes"] * 4 / 1e9, sub[0]["w_bytes"] / 1e9],
          color=["#C44E52", "#4C72B0"])
ax[1].bar(["KV@fp16"], [sub[0]["kv_bytes"] / 1e9], color="#DD8452")
ax[1].set_ylabel("GB"); ax[1].set_title("[C] batch=8, seq=8192")
for i, v in enumerate([sub[0]["w_bytes"] * 4 / 1e9, sub[0]["w_bytes"] / 1e9, sub[0]["kv_bytes"] / 1e9]):
    ax[1].text(i, v, f"{v:.1f}GB", ha="center", va="bottom", fontsize=8)
savefig(fig, "kv_cache_memory_ledger.png")

[C] 显存账本（模型：L=32, d=4096, heads=32；权重约 2.15 B 参数）
 batch     seq      权重@4bit      KV@fp16     KV/权重
     1    2048       1.0 GB     512.0 MB     0.50x
     1    8192       1.0 GB       2.0 GB     2.00x
     1   32768       1.0 GB       8.0 GB     8.00x
     8    2048       1.0 GB       4.0 GB     4.00x
     8    8192       1.0 GB      16.0 GB    16.00x
     8   32768       1.0 GB      64.0 GB    64.00x
    32    2048       1.0 GB      16.0 GB    16.00x
    32    8192       1.0 GB      64.0 GB    64.00x
    32   32768       1.0 GB     256.0 GB   256.00x
  batch=1/seq=2048：KV 是权重(4-bit)的 0.50x
  batch=32/seq=32768：KV 是权重的 256.00x —— 这时权重位宽已经不是主要矛盾
  读数：权重量化省的是「一次性」的显存与带宽，KV cache 随 batch×seq 线性增长；
        长上下文 + 大 batch 下，KV 才是瓶颈 —— 这就是 24 篇的核心论点。


[save] /Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/kv_cache_quant/results/kv_cache_memory_ledger.png


'/Users/congyuan/Desktop/Projects/sandbox/ipynbs/experiments/quantization/kv_cache_quant/results/kv_cache_memory_ledger.png'

In [6]:
summary = {"meta": dict(mode=MODE, cfg={k: (list(v) if isinstance(v, (tuple, list)) else v)
                                          for k, v in CFG.items()}, seed=SEED),
           "A_sensitivity": rows_A, "B_sequence": rows_B,
           "C_ledger": [{k: (float(v) if isinstance(v, (int, float)) else v) for k, v in r.items()}
                        for r in rows_C]}
log("")
log("=" * 84)
log("结论汇总")
log(f"1) [A] {b4}-bit：KV 同时量化时 per-tensor {pt['out']:.3e} -> per-channel {pc['out']:.3e}"
    f"（{10*np.log10(pt['out']/pc['out']):+.2f} dB）")
for b in (b4,):
    for gran in ("per-channel",):
        g = lambda t: [r["out"] for r in rows_A if r["bits"] == b and r["gran"] == gran and r["target"] == t][0]
        log(f"   {b}-bit/{gran}：V 的误差是 K 的 {g('V')/g('K'):.2f} 倍（V 更敏感）")
log(f"2) [B] seq 从 {CFG['seqs'][0]} 到 {CFG['seqs'][-1]}：{b4}-bit 输出误差 "
    f"{[r for r in rows_B if r['bits']==b4][0]['out']:.3e} -> "
    f"{[r for r in rows_B if r['bits']==b4][-1]['out']:.3e}")
log(f"3) [C] batch=1/seq=2048 时 KV 是 W4 权重的 {r1['ratio']:.2f}x；"
    f"batch=32/seq=32768 时 {r2['ratio']:.2f}x —— 长上下文大 batch 下 KV 才是瓶颈")
log("=" * 84)
with open(os.path.join(RES, "results.json"), "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False, default=float)
with open(os.path.join(RES, "stdout.txt"), "w") as f:
    f.write("\n".join(_LINES) + "\n")
log("[save] results.json / stdout.txt")


结论汇总
1) [A] 4-bit：KV 同时量化时 per-tensor 5.817e-01 -> per-channel 6.344e-02（+9.62 dB）
   4-bit/per-channel：V 的误差是 K 的 0.36 倍（V 更敏感）
2) [B] seq 从 128 到 512：4-bit 输出误差 4.987e-02 -> 6.344e-02
3) [C] batch=1/seq=2048 时 KV 是 W4 权重的 0.50x；batch=32/seq=32768 时 256.00x —— 长上下文大 batch 下 KV 才是瓶颈


[save] results.json / stdout.txt
